In [20]:
import nltk
from nltk.util import ngrams
from collections import Counter

In [21]:
text = '오늘은 날씨가 맑다. 오늘은 기분이 좋다. 오늘은 사람이 많다. 오늘은 수업이 좋다.'

In [22]:
tokens = nltk.word_tokenize(text)
tokens

['오늘은',
 '날씨가',
 '맑다',
 '.',
 '오늘은',
 '기분이',
 '좋다',
 '.',
 '오늘은',
 '사람이',
 '많다',
 '.',
 '오늘은',
 '수업이',
 '좋다',
 '.']

In [23]:
unigram = tokens
bigram = list(ngrams(tokens, 2))

unigram_freq = Counter(unigram)
bigram_freq = Counter(bigram)

print(unigram_freq)
print(bigram_freq)

Counter({'오늘은': 4, '.': 4, '좋다': 2, '날씨가': 1, '맑다': 1, '기분이': 1, '사람이': 1, '많다': 1, '수업이': 1})
Counter({('.', '오늘은'): 3, ('좋다', '.'): 2, ('오늘은', '날씨가'): 1, ('날씨가', '맑다'): 1, ('맑다', '.'): 1, ('오늘은', '기분이'): 1, ('기분이', '좋다'): 1, ('오늘은', '사람이'): 1, ('사람이', '많다'): 1, ('많다', '.'): 1, ('오늘은', '수업이'): 1, ('수업이', '좋다'): 1})


In [24]:
for (w1, w2), freq in bigram_freq.items():
    prob = freq / unigram_freq[w1]
    print(f'P({w2} | {w1}) = {prob:.3f}')

P(날씨가 | 오늘은) = 0.250
P(맑다 | 날씨가) = 1.000
P(. | 맑다) = 1.000
P(오늘은 | .) = 0.750
P(기분이 | 오늘은) = 0.250
P(좋다 | 기분이) = 1.000
P(. | 좋다) = 1.000
P(사람이 | 오늘은) = 0.250
P(많다 | 사람이) = 1.000
P(. | 많다) = 1.000
P(수업이 | 오늘은) = 0.250
P(좋다 | 수업이) = 1.000


In [25]:
import math

def compute_bigram_perplexity(test_text, unigram_freq, bigram_freq):
    test_tokens = nltk.word_tokenize(test_text)
    test_bigrams = list(ngrams(test_tokens, 2))

    log_prob_sum = 0
    N = len(test_bigrams)

    for bigram in test_bigrams:
        w1, w2 = bigram
        prob = bigram_freq.get(bigram, 0) / unigram_freq.get(w1, 1)
        if prob == 0:
            prob = 1e-10
        log_prob_sum += math.log2(prob)

    cross_entropy = -log_prob_sum / N
    perplexity = math.pow(2, cross_entropy)

    return perplexity

In [26]:
train_text = '자연어 처리는 재미있다. 자연어 처리는 어렵지만 도전하고 싶다. 오늘은 날씨가 좋다.'

train_tokens = nltk.word_tokenize(train_text)

unigrams = train_tokens
bigrams = list(ngrams(train_tokens, 2))

unigrams_freq = Counter(unigrams)
bigrams_freq = Counter(bigrams)

In [28]:
test_sentences = [
    '자연어 처리는 재미있다.',
    '자연어 처리는 어렵지만 도전하고 싶다.',
    '오늘은 날씨가 좋다.',
    '오늘은 날씨가 맑다.',
    '오늘은 기분이 좋다.',
    '오늘은 사람이 많다.',
    '오늘은 수업이 좋다.'
]

for sentence in test_sentences:
    pp = compute_bigram_perplexity(sentence, unigrams_freq, bigrams_freq)
    print(f'{sentence} Perplexity: {pp}')

자연어 처리는 재미있다. Perplexity: 1.2599210498948732
자연어 처리는 어렵지만 도전하고 싶다. Perplexity: 1.148698354997035
오늘은 날씨가 좋다. Perplexity: 1.0
오늘은 날씨가 맑다. Perplexity: 4641588.833612777
오늘은 기분이 좋다. Perplexity: 4641588.833612777
오늘은 사람이 많다. Perplexity: 10000000000.000008
오늘은 수업이 좋다. Perplexity: 4641588.833612777


In [30]:
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.util import bigrams

train_tokens = [['I', 'love', 'NLP'], ['I', 'love', 'Python']]

train_data, vocab = padded_everygram_pipeline(2, train_tokens)

model = MLE(2)
model.fit(train_data, vocab)

test_tokens = nltk.word_tokenize('I love Python')
test_bigrams = list(bigrams(test_tokens))

perplexity = model.perplexity(test_bigrams)
perplexity

1.4142135623730951